<h1>Calculate Normalized mean-squeared error</h1>

In [1]:
import sys
import os

# Add the path to the `simulated_recons/` folder to the Python path
sys.path.append(os.path.abspath("../../.."))
import livedifferences

from ptypy import io
from ptypy.utils import rmphaseramp
import numpy as np
import re
import glob
import matplotlib.pyplot as plt
%matplotlib widget
plt.ion()


WARNING ptypy - Message Passaging for Python (mpi4py) not found.
    CPU-parallelization disabled.
    Install python-mpi4py via the package repositories or with `pip install --user mpi4py`


<h2>Load data</h2>

In [2]:
ol_cases = {'10px': {'offline_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step10px_1e+10_poisTRUE_spiral_00/simg_startframe400__fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1840.ptyr', 
                     'realtime_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step10px_1e+10_poisTRUE_spiral_00/simg_startframe1____fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1840.ptyr', 
                     'samplename': '_spiralstep10px_1e10_apert-pr-update'}, 
            '19px': {'offline_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step19px_1e+10_poisTRUE_spiral_00/simg_startframe315__fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1710.ptyr', 
                     'realtime_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step19px_1e+10_poisTRUE_spiral_00/simg_startframe1____fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1710.ptyr', 
                     'samplename': '_spiralstep19px_1e10_apert-pr-update'}, 
            '27px': {'offline_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step27px_1e+10_poisTRUE_spiral_00/simg_startframe315__fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1660.ptyr', 
                     'realtime_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step27px_1e+10_poisTRUE_spiral_00/simg_startframe1____fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1660.ptyr', 
                     'samplename': '_spiralstep27px_1e10_apert-pr-update'}, 
            '35px': {'offline_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step35px_1e+10_poisTRUE_spiral_00/simg_startframe315__fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1570.ptyr', 
                     'realtime_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step35px_1e+10_poisTRUE_spiral_00/simg_startframe1____fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1570.ptyr', 
                     'samplename': '_spiralstep35px_1e10_apert-pr-update'}, 
            '40px': {'offline_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step40px_1e+10_poisTRUE_spiral_00/simg_startframe315__fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1530.ptyr', 
                     'realtime_fname': '/data/staff/nanomax/reblex/data-simulated-recons/Siemens-img/simulated_recons/from_simg_256px_Au-Si3N4_step40px_1e+10_poisTRUE_spiral_00/simg_startframe1____fpb25_aperture-pr-update_00/dumps/dump_scan_000000_DM_pycuda_1530.ptyr', 
                     'samplename': '_spiralstep40px_1e10_apert-pr-update'} }


LD10 = livedifferences.LoadData(offline_fname=ol_cases['10px']['offline_fname'], realtime_fname=ol_cases['10px']['realtime_fname'], samplename=ol_cases['10px']['samplename'])
LD19 = livedifferences.LoadData(offline_fname=ol_cases['19px']['offline_fname'], realtime_fname=ol_cases['19px']['realtime_fname'], samplename=ol_cases['19px']['samplename'])
LD27 = livedifferences.LoadData(offline_fname=ol_cases['27px']['offline_fname'], realtime_fname=ol_cases['27px']['realtime_fname'], samplename=ol_cases['27px']['samplename'])
LD35 = livedifferences.LoadData(offline_fname=ol_cases['35px']['offline_fname'], realtime_fname=ol_cases['35px']['realtime_fname'], samplename=ol_cases['35px']['samplename'])
LD40 = livedifferences.LoadData(offline_fname=ol_cases['40px']['offline_fname'], realtime_fname=ol_cases['40px']['realtime_fname'], samplename=ol_cases['40px']['samplename'])

LD10.load()
LD19.load()
LD27.load()
LD35.load()
LD40.load()


In [3]:
LD = LD10
LD.process_data() # Gives obj1_abs, obj2_abs, obj1_abslog, obj2_abslog, obj1_phase, obj2_phase, w1, w2, obj1_ramp, ramp1, obj2_ramp, ramp2, obj1_phaseramp, obj2_phaseramp, obj1_phaseramp_unwrapped, obj2_phaseramp_unwrapped, obj1_rmramp_abs, obj2_rmramp_abs, obj1_rmramp_abslog, obj2_rmramp_abslog, diff_rmramp_abslog, diff_abslog_od, diff_phase, diff_phaseramp, diff_phaseramp_unwrapped, diff_phaseramp_unwrapped_ramp, shift1, error, phasediff
LD.load_GT() # Gives fname3, fname_pr, pr, obj, obj_abs, obj_abslog, obj_phase, w, obj_ramp, ramp, obj_rmramp_abs, obj_rmramp_abslog, obj_phaseramp, obj_phaseramp_unwrapped
LD.add_padding() # Gives sh, sh1, padrow, padcol, obj1_pad, obj2_pad, obj1_abs_pad, obj2_abs_pad, obj1_abslog_pad, obj2_abslog_pad, obj1_phase_pad, obj2_phase_pad, obj1_ramp_pad, obj2_ramp_pad, obj1_rmramp_abs_pad, obj2_rmramp_abs_pad, obj1_rmramp_abslog_pad, obj2_rmramp_abslog_pad, obj1_phaseramp_pad, obj2_phaseramp_pad, obj1_phaseramp_unwrapped_pad, obj2_phaseramp_unwrapped_pad
print(LD.sh, LD.sh1, LD.obj1_phaseramp_unwrapped_pad.shape)
########
LD10.process_data() # Gives obj1_abs, obj2_abs, obj1_abslog, obj2_abslog, obj1_phase, obj2_phase, w1, w2, obj1_ramp, ramp1, obj2_ramp, ramp2, obj1_phaseramp, obj2_phaseramp, obj1_phaseramp_unwrapped, obj2_phaseramp_unwrapped, obj1_rmramp_abs, obj2_rmramp_abs, obj1_rmramp_abslog, obj2_rmramp_abslog, diff_rmramp_abslog, diff_abslog_od, diff_phase, diff_phaseramp, diff_phaseramp_unwrapped, diff_phaseramp_unwrapped_ramp, shift1, error, phasediff
LD10.load_GT() # Gives fname3, fname_pr, pr, obj, obj_abs, obj_abslog, obj_phase, w, obj_ramp, ramp, obj_rmramp_abs, obj_rmramp_abslog, obj_phaseramp, obj_phaseramp_unwrapped
LD10.add_padding() # Gives sh, sh1, padrow, padcol, obj1_pad, obj2_pad, obj1_abs_pad, obj2_abs_pad, obj1_abslog_pad, obj2_abslog_pad, obj1_phase_pad, obj2_phase_pad, obj1_ramp_pad, obj2_ramp_pad, obj1_rmramp_abs_pad, obj2_rmramp_abs_pad, obj1_rmramp_abslog_pad, obj2_rmramp_abslog_pad, obj1_phaseramp_pad, obj2_phaseramp_pad, obj1_phaseramp_unwrapped_pad, obj2_phaseramp_unwrapped_pad
print(LD10.sh, LD10.sh1, LD10.obj1_phaseramp_unwrapped_pad.shape)

LD19.process_data() # Gives obj1_abs, obj2_abs, obj1_abslog, obj2_abslog, obj1_phase, obj2_phase, w1, w2, obj1_ramp, ramp1, obj2_ramp, ramp2, obj1_phaseramp, obj2_phaseramp, obj1_phaseramp_unwrapped, obj2_phaseramp_unwrapped, obj1_rmramp_abs, obj2_rmramp_abs, obj1_rmramp_abslog, obj2_rmramp_abslog, diff_rmramp_abslog, diff_abslog_od, diff_phase, diff_phaseramp, diff_phaseramp_unwrapped, diff_phaseramp_unwrapped_ramp, shift1, error, phasediff
LD19.load_GT() # Gives fname3, fname_pr, pr, obj, obj_abs, obj_abslog, obj_phase, w, obj_ramp, ramp, obj_rmramp_abs, obj_rmramp_abslog, obj_phaseramp, obj_phaseramp_unwrapped
LD19.add_padding() # Gives sh, sh1, padrow, padcol, obj1_pad, obj2_pad, obj1_abs_pad, obj2_abs_pad, obj1_abslog_pad, obj2_abslog_pad, obj1_phase_pad, obj2_phase_pad, obj1_ramp_pad, obj2_ramp_pad, obj1_rmramp_abs_pad, obj2_rmramp_abs_pad, obj1_rmramp_abslog_pad, obj2_rmramp_abslog_pad, obj1_phaseramp_pad, obj2_phaseramp_pad, obj1_phaseramp_unwrapped_pad, obj2_phaseramp_unwrapped_pad
print(LD19.sh, LD19.sh1, LD19.obj1_phaseramp_unwrapped_pad.shape)

LD27.process_data() # Gives obj1_abs, obj2_abs, obj1_abslog, obj2_abslog, obj1_phase, obj2_phase, w1, w2, obj1_ramp, ramp1, obj2_ramp, ramp2, obj1_phaseramp, obj2_phaseramp, obj1_phaseramp_unwrapped, obj2_phaseramp_unwrapped, obj1_rmramp_abs, obj2_rmramp_abs, obj1_rmramp_abslog, obj2_rmramp_abslog, diff_rmramp_abslog, diff_abslog_od, diff_phase, diff_phaseramp, diff_phaseramp_unwrapped, diff_phaseramp_unwrapped_ramp, shift1, error, phasediff
LD27.load_GT() # Gives fname3, fname_pr, pr, obj, obj_abs, obj_abslog, obj_phase, w, obj_ramp, ramp, obj_rmramp_abs, obj_rmramp_abslog, obj_phaseramp, obj_phaseramp_unwrapped
LD27.add_padding() # Gives sh, sh1, padrow, padcol, obj1_pad, obj2_pad, obj1_abs_pad, obj2_abs_pad, obj1_abslog_pad, obj2_abslog_pad, obj1_phase_pad, obj2_phase_pad, obj1_ramp_pad, obj2_ramp_pad, obj1_rmramp_abs_pad, obj2_rmramp_abs_pad, obj1_rmramp_abslog_pad, obj2_rmramp_abslog_pad, obj1_phaseramp_pad, obj2_phaseramp_pad, obj1_phaseramp_unwrapped_pad, obj2_phaseramp_unwrapped_pad
print(LD27.sh, LD27.sh1, LD27.obj1_phaseramp_unwrapped_pad.shape)

LD35.process_data() # Gives obj1_abs, obj2_abs, obj1_abslog, obj2_abslog, obj1_phase, obj2_phase, w1, w2, obj1_ramp, ramp1, obj2_ramp, ramp2, obj1_phaseramp, obj2_phaseramp, obj1_phaseramp_unwrapped, obj2_phaseramp_unwrapped, obj1_rmramp_abs, obj2_rmramp_abs, obj1_rmramp_abslog, obj2_rmramp_abslog, diff_rmramp_abslog, diff_abslog_od, diff_phase, diff_phaseramp, diff_phaseramp_unwrapped, diff_phaseramp_unwrapped_ramp, shift1, error, phasediff
LD35.load_GT() # Gives fname3, fname_pr, pr, obj, obj_abs, obj_abslog, obj_phase, w, obj_ramp, ramp, obj_rmramp_abs, obj_rmramp_abslog, obj_phaseramp, obj_phaseramp_unwrapped
LD35.add_padding() # Gives sh, sh1, padrow, padcol, obj1_pad, obj2_pad, obj1_abs_pad, obj2_abs_pad, obj1_abslog_pad, obj2_abslog_pad, obj1_phase_pad, obj2_phase_pad, obj1_ramp_pad, obj2_ramp_pad, obj1_rmramp_abs_pad, obj2_rmramp_abs_pad, obj1_rmramp_abslog_pad, obj2_rmramp_abslog_pad, obj1_phaseramp_pad, obj2_phaseramp_pad, obj1_phaseramp_unwrapped_pad, obj2_phaseramp_unwrapped_pad
print(LD35.sh, LD35.sh1, LD35.obj1_phaseramp_unwrapped_pad.shape)

LD40.process_data() # Gives obj1_abs, obj2_abs, obj1_abslog, obj2_abslog, obj1_phase, obj2_phase, w1, w2, obj1_ramp, ramp1, obj2_ramp, ramp2, obj1_phaseramp, obj2_phaseramp, obj1_phaseramp_unwrapped, obj2_phaseramp_unwrapped, obj1_rmramp_abs, obj2_rmramp_abs, obj1_rmramp_abslog, obj2_rmramp_abslog, diff_rmramp_abslog, diff_abslog_od, diff_phase, diff_phaseramp, diff_phaseramp_unwrapped, diff_phaseramp_unwrapped_ramp, shift1, error, phasediff
LD40.load_GT() # Gives fname3, fname_pr, pr, obj, obj_abs, obj_abslog, obj_phase, w, obj_ramp, ramp, obj_rmramp_abs, obj_rmramp_abslog, obj_phaseramp, obj_phaseramp_unwrapped
LD40.add_padding() # Gives sh, sh1, padrow, padcol, obj1_pad, obj2_pad, obj1_abs_pad, obj2_abs_pad, obj1_abslog_pad, obj2_abslog_pad, obj1_phase_pad, obj2_phase_pad, obj1_ramp_pad, obj2_ramp_pad, obj1_rmramp_abs_pad, obj2_rmramp_abs_pad, obj1_rmramp_abslog_pad, obj2_rmramp_abslog_pad, obj1_phaseramp_pad, obj2_phaseramp_pad, obj1_phaseramp_unwrapped_pad, obj2_phaseramp_unwrapped_pad
print(LD40.sh, LD40.sh1, LD40.obj1_phaseramp_unwrapped_pad.shape)


(1341, 1341) (446, 451) (1341, 1341)
(1341, 1341) (446, 451) (1341, 1341)
(1341, 1341) (617, 626) (1341, 1341)
(1341, 1341) (769, 782) (1341, 1341)
(1341, 1341) (921, 938) (1341, 1341)
(1341, 1341) (1016, 1036) (1341, 1341)


In [4]:
LD.pr.shape

(256, 256)

<h2>Calculate</h2>

In [5]:
# Normalized mean-squeared error
from skimage.registration import phase_cross_correlation
from scipy.ndimage import shift

def NMSE(obj1_, obj2_, obj_GT, printoutput=True):
    """Calculated as in eqn (8) and (9) in Maiden, A. M. & Rodenburg, J. M. An improved ptychographical phase retrieval algorithm for diffractive imaging.
Ultramicroscopy 109, 1256–1262 (2009)."""
    W1 = LD.rampweight(obj1_, scale=100)#
    obj1_ = rmphaseramp(obj1_, W1)
    obj1_ = rmphaseramp(obj1_, W1)
    obj1_ = rmphaseramp(obj1_, W1)
    obj1_ = rmphaseramp(obj1_, W1)
    obj1_ = rmphaseramp(obj1_, W1)
    obj2_ = rmphaseramp(obj2_, W1)
    obj2_ = rmphaseramp(obj2_, W1)
    obj2_ = rmphaseramp(obj2_, W1)
    obj2_ = rmphaseramp(obj2_, W1)
    obj2_ = rmphaseramp(obj2_, W1)
    obj_GT = rmphaseramp(obj_GT, W1)
    obj_GT = rmphaseramp(obj_GT, W1)
    obj_GT = rmphaseramp(obj_GT, W1)
    obj_GT = rmphaseramp(obj_GT, W1)
    obj_GT = rmphaseramp(obj_GT, W1)


    gamma1 = ( np.sum( obj_GT*np.conjugate(obj1_) ) ) / (np.sum( np.abs(obj1_)**2 ))
    gamma2 = ( np.sum( obj_GT*np.conjugate(obj2_) ) ) / (np.sum( np.abs(obj2_)**2 ))
    gamma12 = ( np.sum( obj1_*np.conjugate(obj2_) ) ) / (np.sum( np.abs(obj2_)**2 ))
    gamma21 = ( np.sum( obj2_*np.conjugate(obj1_) ) ) / (np.sum( np.abs(obj1_)**2 ))  # Just to ensure the order doesn't matter

    #print(np.abs(gamma1), np.mean(np.abs(obj_GT)/np.abs(obj1_)))
    E1 = ( np.sum( np.abs(obj_GT-gamma1*obj1_)**2 ) ) / ( np.sum( np.abs(obj_GT)**2 ) )
    E2 = ( np.sum( np.abs(obj_GT-gamma2*obj2_)**2 ) ) / ( np.sum( np.abs(obj_GT)**2 ) )
    E12 = ( np.sum( np.abs(obj1_-gamma12*obj2_)**2 ) ) / ( np.sum( np.abs(obj1_)**2 ) )
    E21 = ( np.sum( np.abs(obj2_-gamma21*obj1_)**2 ) ) / ( np.sum( np.abs(obj2_)**2 ) )  # Just to ensure the order doesn't matter
    
    if printoutput:
        print(f'NMSE(obj_GT, obj1_): {E1:.2e}')
        print(f'NMSE(obj_GT, obj2_): {E2:.2e}')
        print(f'NMSE(obj1_,  obj2_): {E12:.2e}')
        print(f'NMSE(obj2_,  obj1_): {E21:.2e}')


    #### Peak signal to noise ratio
    MSE1 = np.mean(np.abs(obj_GT - obj1_)**2)
    MSE2 = np.mean(np.abs(obj_GT - obj2_)**2)
    MSE12 = np.mean(np.abs(obj1_ - obj2_)**2)

    max_val = np.max(np.abs(obj_GT))
    max_val1 = np.max(np.abs(obj1_))

    PSNR1 = 10 * np.log10(max_val**2 / MSE1)
    PSNR2 = 10 * np.log10(max_val**2 / MSE2)
    PSNR12 = 10 * np.log10(max_val1**2 / MSE12)
    
    if printoutput:
        print('PSNR: ', PSNR1, PSNR2, PSNR12)
    return E1, E2, E12

LD = LD10
print('NMSE for cropped object')
marg = ((min(LD.sh) - max(LD.sh1)) // 2) + int(max(LD.sh1) // 3.15)#3.8)
obj1_ = np.array(LD.obj1_ramp_pad[marg:-marg,marg:-marg].copy())
obj2_ = np.array(LD.obj2_ramp_pad[marg:-marg,marg:-marg].copy())
obj_GT = np.array(LD.obj_ramp[marg:-marg,marg:-marg].copy())
NMSE(obj1_, obj2_, obj_GT)

print('\nNMSE for full object')
obj1_full = np.array(LD.obj1_ramp_pad[LD.padrow1:LD.padrow1+LD.sh1[0], LD.padcol1:LD.padcol1+LD.sh1[1]].copy())
obj2_full = np.array(LD.obj2_ramp_pad[LD.padrow2:LD.padrow2+LD.sh1[0], LD.padcol2:LD.padcol2+LD.sh1[1]].copy())
obj_GT_full = np.array(LD.obj_ramp[LD.padrow1:LD.padrow1+LD.sh1[0], LD.padcol1:LD.padcol1+LD.sh1[1]].copy())
NMSE(obj1_full, obj2_full, obj_GT_full)


NMSE for cropped object
NMSE(obj_GT, obj1_): 1.81e-08
NMSE(obj_GT, obj2_): 1.82e-08
NMSE(obj1_,  obj2_): 6.56e-12
NMSE(obj2_,  obj1_): 6.56e-12
PSNR:  -4.684705758113149 -5.034388477788064 11.743612648682038

NMSE for full object
NMSE(obj_GT, obj1_): 4.76e-02
NMSE(obj_GT, obj2_): 4.87e-02
NMSE(obj1_,  obj2_): 1.55e-03
NMSE(obj2_,  obj1_): 1.55e-03
PSNR:  -4.63845747440352 -4.915119849385411 13.737470131696252


(0.04760934035270598, 0.048736726618529826, 0.001548604683588125)

In [6]:
"""NMSE for cropped object
NMSE(obj_GT, obj1_): 1.81e-08
NMSE(obj_GT, obj2_): 1.82e-08
NMSE(obj1_,  obj2_): 6.56e-12
NMSE(obj2_,  obj1_): 6.56e-12
PSNR:  -4.684705758113149 -5.034388477788064 11.743612648682038

NMSE for full object
NMSE(obj_GT, obj1_): 4.76e-02
NMSE(obj_GT, obj2_): 4.87e-02
NMSE(obj1_,  obj2_): 1.55e-03
NMSE(obj2_,  obj1_): 1.55e-03
PSNR:  -4.63845747440352 -4.915119849385411 13.737470131696252
(0.04760934035270598, 0.048736726618529826, 0.001548604683588125)"""

'NMSE for cropped object\nNMSE(obj_GT, obj1_): 1.81e-08\nNMSE(obj_GT, obj2_): 1.82e-08\nNMSE(obj1_,  obj2_): 6.56e-12\nNMSE(obj2_,  obj1_): 6.56e-12\nPSNR:  -4.684705758113149 -5.034388477788064 11.743612648682038\n\nNMSE for full object\nNMSE(obj_GT, obj1_): 4.76e-02\nNMSE(obj_GT, obj2_): 4.87e-02\nNMSE(obj1_,  obj2_): 1.55e-03\nNMSE(obj2_,  obj1_): 1.55e-03\nPSNR:  -4.63845747440352 -4.915119849385411 13.737470131696252\n(0.04760934035270598, 0.048736726618529826, 0.001548604683588125)'

In [7]:
# Calculate NMSE for all overlap reconstructions, cropped to 165x165 pixels

print('NMSE for 165x165 cropped objects')
marg = 588#((min(LD10.sh) - max(LD10.sh1)) // 2) + int(max(LD10.sh1) // 3.15)#3.8)
obj1_10 = np.array(LD10.obj1_ramp_pad[marg:-marg,marg:-marg].copy())
obj2_10 = np.array(LD10.obj2_ramp_pad[marg:-marg,marg:-marg].copy())
obj_GT10 = np.array(LD10.obj_ramp[marg:-marg,marg:-marg].copy())

obj1_19 = np.array(LD19.obj1_ramp_pad[marg:-marg,marg:-marg].copy())
obj2_19 = np.array(LD19.obj2_ramp_pad[marg:-marg,marg:-marg].copy())
obj_GT19 = np.array(LD19.obj_ramp[marg:-marg,marg:-marg].copy())
###
obj1_27 = np.array(LD27.obj1_ramp_pad[marg:-marg,marg:-marg].copy())
obj2_27 = np.array(LD27.obj2_ramp_pad[marg:-marg,marg:-marg].copy())
obj_GT27 = np.array(LD27.obj_ramp[marg:-marg,marg:-marg].copy())
###
obj1_35 = np.array(LD35.obj1_ramp_pad[marg:-marg,marg:-marg].copy())
obj2_35 = np.array(LD35.obj2_ramp_pad[marg:-marg,marg:-marg].copy())
obj_GT35 = np.array(LD35.obj_ramp[marg:-marg,marg:-marg].copy())
###
obj1_40 = np.array(LD40.obj1_ramp_pad[marg:-marg,marg:-marg].copy())
obj2_40 = np.array(LD40.obj2_ramp_pad[marg:-marg,marg:-marg].copy())
obj_GT40 = np.array(LD40.obj_ramp[marg:-marg,marg:-marg].copy())

E1_10, E2_10, E12_10 = NMSE(obj1_10, obj2_10, obj_GT10, printoutput=False)
E1_19, E2_19, E12_19 = NMSE(obj1_19, obj2_19, obj_GT19, printoutput=False)
E1_27, E2_27, E12_27 = NMSE(obj1_27, obj2_27, obj_GT27, printoutput=False)
E1_35, E2_35, E12_35 = NMSE(obj1_35, obj2_35, obj_GT35, printoutput=False)
E1_40, E2_40, E12_40 = NMSE(obj1_40, obj2_40, obj_GT40, printoutput=False)

print('10 px')
print(f'NMSE(Offline - Ground truth): {E1_10:.2e}')
print(f'NMSE(Real-time - Ground truth): {E2_10:.2e}')
print(f'NMSE(Real-time - Offline): {E12_10:.2e}\n')

print('19 px')
print(f'NMSE(Offline - Ground truth): {E1_19:.2e}')
print(f'NMSE(Real-time - Ground truth): {E2_19:.2e}')
print(f'NMSE(Real-time - Offline): {E12_19:.2e}\n')

print('27 px')
print(f'NMSE(Offline - Ground truth): {E1_27:.2e}')
print(f'NMSE(Real-time - Ground truth): {E2_27:.2e}')
print(f'NMSE(Real-time - Offline): {E12_27:.2e}\n')

print('35 px')
print(f'NMSE(Offline - Ground truth): {E1_35:.2e}')
print(f'NMSE(Real-time - Ground truth): {E2_35:.2e}')
print(f'NMSE(Real-time - Offline): {E12_35:.2e}\n')

print('40 px')
print(f'NMSE(Offline - Ground truth): {E1_40:.2e}')
print(f'NMSE(Real-time - Ground truth): {E2_40:.2e}')
print(f'NMSE(Real-time - Offline): {E12_40:.2e}\n')

NMSE for 165x165 cropped objects
10 px
NMSE(Offline - Ground truth): 1.81e-08
NMSE(Real-time - Ground truth): 1.82e-08
NMSE(Real-time - Offline): 6.56e-12

19 px
NMSE(Offline - Ground truth): 5.43e-08
NMSE(Real-time - Ground truth): 5.43e-08
NMSE(Real-time - Offline): 1.83e-11

27 px
NMSE(Offline - Ground truth): 1.13e-07
NMSE(Real-time - Ground truth): 1.12e-07
NMSE(Real-time - Offline): 4.29e-11

35 px
NMSE(Offline - Ground truth): 1.97e-07
NMSE(Real-time - Ground truth): 1.94e-07
NMSE(Real-time - Offline): 5.09e-09

40 px
NMSE(Offline - Ground truth): 3.41e-07
NMSE(Real-time - Ground truth): 3.60e-07
NMSE(Real-time - Offline): 2.73e-08



In [8]:
# Save this data to file
np.save("NMSE_data.npy", np.array([[E1_10, E2_10, E12_10], [E1_19, E2_19, E12_19], [E1_27, E2_27, E12_27], [E1_35, E2_35, E12_35], [E1_40, E2_40, E12_40]]).T)
print('saved data')


saved data


<h2>Extra: Calculate as a funtion of different croppings</h2>

In [9]:
print(LD10.sh1, LD19.sh1, LD27.sh1, LD35.sh1, LD40.sh1)

obj_cropped10 = LD10.obj[(LD10.sh[0]-LD10.sh1[0])//2:-(LD10.sh[0]-LD10.sh1[0])//2, (LD10.sh[1]-LD10.sh1[1])//2:-(LD10.sh[1]-LD10.sh1[1])//2].copy()
obj_cropped19 = LD19.obj[(LD19.sh[0]-LD19.sh1[0])//2:-(LD19.sh[0]-LD19.sh1[0])//2, (LD19.sh[1]-LD19.sh1[1])//2:-(LD19.sh[1]-LD19.sh1[1])//2].copy()
obj_cropped27 = LD27.obj[(LD27.sh[0]-LD27.sh1[0])//2:-(LD27.sh[0]-LD27.sh1[0])//2, (LD27.sh[1]-LD27.sh1[1])//2:-(LD27.sh[1]-LD27.sh1[1])//2].copy()
obj_cropped35 = LD35.obj[(LD35.sh[0]-LD35.sh1[0])//2:-(LD35.sh[0]-LD35.sh1[0])//2, (LD35.sh[1]-LD35.sh1[1])//2:-(LD35.sh[1]-LD35.sh1[1])//2].copy()
obj_cropped40 = LD40.obj[(LD40.sh[0]-LD40.sh1[0])//2:-(LD40.sh[0]-LD40.sh1[0])//2, (LD40.sh[1]-LD40.sh1[1])//2:-(LD40.sh[1]-LD40.sh1[1])//2].copy()
print(obj_cropped10.shape, obj_cropped19.shape, obj_cropped27.shape, obj_cropped35.shape, obj_cropped40.shape)

E_10 = []
shapes10 = []
margs = np.arange(10, np.min(LD10.sh1), 10)//2#((min(LD.sh) - max(LD.sh1)) // 2) + int(max(LD.sh1) // 3.15)#3.8)
for marg in margs:
    obj1_ = np.array(LD10.obj1[marg:-marg,marg:-marg].copy())
    obj2_ = np.array(LD10.obj2[marg:-marg,marg:-marg].copy())
    obj_GT = np.array(obj_cropped10[marg:-marg,marg:-marg].copy())
    E_10.append(NMSE(obj1_, obj2_, obj_GT, printoutput=False))
    shapes10.append(obj1_.shape)

E_19 = []
shapes19 = []
margs = np.arange(10, np.min(LD19.sh1), 10)//2#((min(LD.sh) - max(LD.sh1)) // 2) + int(max(LD.sh1) // 3.15)#3.8)
for marg in margs:
    obj1_ = np.array(LD19.obj1[marg:-marg,marg:-marg].copy())
    obj2_ = np.array(LD19.obj2[marg:-marg,marg:-marg].copy())
    obj_GT = np.array(obj_cropped19[marg:-marg,marg:-marg].copy())
    E_19.append(NMSE(obj1_, obj2_, obj_GT, printoutput=False))
    shapes19.append(obj1_.shape)
    
E_27 = []
shapes27 = []
margs = np.arange(10, np.min(LD27.sh1), 10)//2#((min(LD.sh) - max(LD.sh1)) // 2) + int(max(LD.sh1) // 3.15)#3.8)
for marg in margs:
    obj1_ = np.array(LD27.obj1[marg:-marg,marg:-marg].copy())
    obj2_ = np.array(LD27.obj2[marg:-marg,marg:-marg].copy())
    obj_GT = np.array(obj_cropped27[marg:-marg,marg:-marg].copy())
    E_27.append(NMSE(obj1_, obj2_, obj_GT, printoutput=False))
    shapes27.append(obj1_.shape)
    
E_35 = []
shapes35 = []
margs = np.arange(10, np.min(LD35.sh1)-1, 10)//2#((min(LD.sh) - max(LD.sh1)) // 2) + int(max(LD.sh1) // 3.15)#3.8)
for marg in margs:
    obj1_ = np.array(LD35.obj1[marg:-marg,marg:-marg].copy())
    obj2_ = np.array(LD35.obj2[marg:-marg,marg:-marg].copy())
    obj_GT = np.array(obj_cropped35[marg:-marg,marg:-marg].copy())
    E_35.append(NMSE(obj1_, obj2_, obj_GT, printoutput=False))
    shapes35.append(obj1_.shape)
    
E_40 = []
shapes40 = []
margs = np.arange(10, np.min(LD40.sh1)-1, 10)//2#((min(LD.sh) - max(LD.sh1)) // 2) + int(max(LD.sh1) // 3.15)#3.8)
for marg in margs:
    obj1_ = np.array(LD40.obj1[marg:-marg,marg:-marg].copy())
    obj2_ = np.array(LD40.obj2[marg:-marg,marg:-marg].copy())
    obj_GT = np.array(obj_cropped40[marg:-marg,marg:-marg].copy())
    E_40.append(NMSE(obj1_, obj2_, obj_GT, printoutput=False))
    shapes40.append(obj1_.shape)

(446, 451) (617, 626) (769, 782) (921, 938) (1016, 1036)
(446, 451) (617, 626) (769, 782) (921, 938) (1016, 1036)


In [10]:
E_10 = np.array(E_10)
E_19 = np.array(E_19)
E_27 = np.array(E_27)
E_35 = np.array(E_35)
E_40 = np.array(E_40)
shapes10 = np.array(shapes10)
shapes19 = np.array(shapes19)
shapes27 = np.array(shapes27)
shapes35 = np.array(shapes35)
shapes40 = np.array(shapes40)

plt.figure()
off_GT10=plt.plot(shapes10[:,0],E_10[:,0],'o-', c='r')
RT_GT10=plt.plot(shapes10[:,0],E_10[:,1],'.-', c='r')
#off_RT10=plt.plot(shapes10[:,0],E_10[:,2],'d-', c='r')
plt.legend(['Offline - GT', 'Real-time - GT', 'Offline - Real-time'])

off_GT19=plt.plot(shapes19[:,0],E_19[:,0],'o-', c='b')
RT_GT19=plt.plot(shapes19[:,0],E_19[:,1],'.-', c='b')
#off_RT19=plt.plot(shapes19[:,0],E_19[:,2],'d-', c='b')

off_GT27=plt.plot(shapes27[:,0],E_27[:,0],'o-', c='c')
RT_GT27=plt.plot(shapes27[:,0],E_27[:,1],'.-', c='c')
#off_RT27=plt.plot(shapes27[:,0],E_27[:,2],'d-', c='c')

off_GT35=plt.plot(shapes35[:,0],E_35[:,0],'o-', c='g')
RT_GT35=plt.plot(shapes35[:,0],E_35[:,1],'.-', c='g')
#off_RT35=plt.plot(shapes35[:,0],E_35[:,2],'d-', c='g')

off_GT40=plt.plot(shapes40[:,0],E_40[:,0],'o-', c='m')
RT_GT40=plt.plot(shapes40[:,0],E_40[:,1],'.-', c='m', mec='k')
#off_RT40=plt.plot(shapes40[:,0],E_40[:,2],'d-', c='m')
#plt.legend((off_GT10, off_GT19, off_GT27, off_GT35, off_GT40), ['10', '19', '27', '35', '40'], loc='upper left')
plt.xlabel('Cropped shape size')
plt.ylabel('NMSE')
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …